## Overview
- This notebook standardizes and cleans asset data using Barker's Source of Truth conventions.
- It supports both:
  - single-source processing (no merge), and
  - daily + historical merge workflows.


In [1]:
from SoT_Cleaner import (
    load_auction_sales_by_asset,
    load_historical_training_data,
    ArtworkPreprocessor,
    WatchPreprocessor,
    DiamondPreprocessor,
    concat_same_schema,
    SoTDataCleaner,
)

import pandas as pd

asset_type = "Art"

# Pipeline mode:
# - "single_source": process one source only (no historical merge)
# - "daily_plus_historical": merge processed source data with historical data
pipeline_mode = "single_source"

# Preprocessor to use: "artwork", "watch", or "diamond"
preprocessor_type = "artwork"

#Set Output file name
output_fname = "artwork_abs_dataset_sot"

# If False, source data is passed through without asset preprocessing
apply_source_preprocessing = True

# If source JSON has no split column, treat source rows as either:
# "non_sot" or "sot"
single_source_role = "non_sot"

# Choose where source data comes from: "bucket" or "json"
load_mode = "json"

# Used when load_mode == "bucket"
bucket_name = "rag-artworks-jsons-ajay-final"

# Used when load_mode == "json"
json_path = "Data/Input/artwork_abs_dataset.json"
json_split_col = "auction_url"  # split SOT vs non-SOT when available

# Used only when pipeline_mode == "daily_plus_historical"
historical_bucket_name = "watch_training_dataset"
historical_identifier = "restandardized"

# Outlier handling in final output
remove_outliers_in_final = False

PREPROCESSOR_MAP = {
    "artwork": ArtworkPreprocessor,
    "art": ArtworkPreprocessor,
    "watch": WatchPreprocessor,
    "diamond": DiamondPreprocessor,
}

if preprocessor_type.lower() not in PREPROCESSOR_MAP:
    raise ValueError(f"No available preprocessor for {preprocessor_type}.")

PreprocessorClass = PREPROCESSOR_MAP[preprocessor_type.lower()]
print(f"Using preprocessor: {PreprocessorClass.__name__}")
print(f"Pipeline mode: {pipeline_mode}")


Using preprocessor: ArtworkPreprocessor
Pipeline mode: single_source


# I. Extraction

## Load auction data

In [2]:
# 1. Load source data
if load_mode == "bucket":
    non_sot_df, sot_df = load_auction_sales_by_asset(
        bucket_name=bucket_name,
        asset_type=asset_type,
    )
elif load_mode == "json":
    source_df = pd.read_json(json_path)
    print(f"Loaded {len(source_df)} rows from {json_path}")

    if json_split_col in source_df.columns:
        mask = source_df[json_split_col].notna()
        sot_df = source_df[mask].copy()
        non_sot_df = source_df[~mask].copy()
        print(f"Split by `{json_split_col}` -> SOT: {len(sot_df)}, Non-SOT: {len(non_sot_df)}")
    else:
        if single_source_role == "sot":
            sot_df = source_df.copy()
            non_sot_df = pd.DataFrame(columns=source_df.columns)
        elif single_source_role == "non_sot":
            non_sot_df = source_df.copy()
            sot_df = pd.DataFrame(columns=source_df.columns)
        else:
            raise ValueError("single_source_role must be either 'non_sot' or 'sot'")

        print(f"`{json_split_col}` not found. Treating all rows as {single_source_role}.")
else:
    raise ValueError("load_mode must be either 'bucket' or 'json'")


Loaded 350378 rows from Data/Input/artwork_abs_dataset.json
`auction_url` not found. Treating all rows as non_sot.


## Preprocess

In [3]:
# 2. Preprocess source data
def preprocess_source_df(df, role: str, label: str) -> pd.DataFrame:
    if df is None or df.empty:
        print(f"Skipping {label}: 0 rows")
        return pd.DataFrame()

    print(f"\nPROCESSING {label}...")

    if not apply_source_preprocessing:
        out = df.copy()
        if role == "non_sot" and "auction_url" not in out.columns:
            out["auction_url"] = None
        print(f"Preprocessing disabled. Shape: {out.shape}")
        return out

    processor = PreprocessorClass(df)
    if role == "sot":
        out = processor.preprocess_sot().get_df()
    elif role == "non_sot":
        out = processor.preprocess_non_sot({"auction_url": None}).get_df()
    else:
        raise ValueError("role must be 'sot' or 'non_sot'")

    print(f"Processed shape ({label}): {out.shape}")
    return out

non_sot_processed = preprocess_source_df(non_sot_df, role="non_sot", label="NON-SOT SOURCE")
sot_processed = preprocess_source_df(sot_df, role="sot", label="SOT SOURCE")

# 3. Combine source parts only when both exist
if not non_sot_processed.empty and not sot_processed.empty:
    df_temp = concat_same_schema(
        sot_processed,
        non_sot_processed,
        left_name="SOT",
        right_name="Non-SOT",
    )
elif not non_sot_processed.empty:
    df_temp = non_sot_processed.copy()
    print("Using Non-SOT source only (no SOT merge needed).")
elif not sot_processed.empty:
    df_temp = sot_processed.copy()
    print("Using SOT source only (no Non-SOT merge needed).")
else:
    raise ValueError("No source rows available after loading/preprocessing")

print(f"Source dataset shape: {df_temp.shape}")



PROCESSING NON-SOT SOURCE...

[1] Convert columns to datetime
  Columns converted (1): ['sale_date']
  Rows affected: 350378

[2] Normalize null-like values
  Rows affected: 350378 → 350378
  New nulls added: 271301

[3] Build `artist_name` from `artist_first_name` + `artist_last_name`
  Column created/updated: artist_name
  Rows affected: 350378

[4] Drop columns
  Columns dropped (1): ['medium']
  Rows affected: 350378 → 350378

[5] Rename columns
  Columns renamed (1): {'medium_type': 'medium'}
  Rows affected: 350378

[6] Rename columns
  Columns renamed (12): {'sale_number': 'sale_id', 'artwork_ah_low_estimate': 'ah_low_estimate', 'artwork_ah_high_estimate': 'ah_high_estimate', 'artwork_ah_bp_price': 'ah_bp_price', 'hammer_price_original': 'ah_price', 'artwork_ah_low_estimate_usd': 'ah_low_estimate_usd', 'artwork_ah_high_estimate_usd': 'ah_high_estimate_usd', 'artwork_ah_price_usd': 'ah_bp_price_usd', 'hammer_price_usd': 'ah_price_usd', 'title': 'artwork_title', 'description': 'd

## Load historical (optional)

In [4]:
# 4. Load historical (optional)
if pipeline_mode == "daily_plus_historical":
    historical_df = load_historical_training_data(
        bucket_name=historical_bucket_name,
        identifier=historical_identifier,
    )

    print("\nPROCESSING HISTORICAL DATA...")
    if apply_source_preprocessing:
        historical_processed = (
            PreprocessorClass(historical_df)
                .preprocess_non_sot({"auction_url": None})
                .get_df()
        )
    else:
        historical_processed = historical_df.copy()
        if "auction_url" not in historical_processed.columns:
            historical_processed["auction_url"] = None

    print(f"Historical dataset shape: {historical_processed.shape}")
else:
    historical_processed = pd.DataFrame()
    print("Skipping historical load because pipeline_mode='single_source'.")


Skipping historical load because pipeline_mode='single_source'.


## Final dataset assembly

In [5]:
# 5. Final dataset assembly
if pipeline_mode == "daily_plus_historical":
    if historical_processed.empty:
        raise ValueError("pipeline_mode=daily_plus_historical requires non-empty historical data")

    final_df = concat_same_schema(
        df_temp,
        historical_processed,
        left_name="SOURCE",
        right_name="HISTORICAL",
    )
    print("Merged source and historical datasets.")
elif pipeline_mode == "single_source":
    final_df = df_temp.copy()
    print("Using single-source dataset. Historical merge skipped.")
else:
    raise ValueError("pipeline_mode must be either 'single_source' or 'daily_plus_historical'")

final_df.info()


Using single-source dataset. Historical merge skipped.
<class 'pandas.core.frame.DataFrame'>
Index: 146915 entries, 0 to 146916
Data columns (total 39 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   asset_type            146915 non-null  object        
 1   auction_house         146915 non-null  object        
 2   sale_id               146914 non-null  object        
 3   sale_date             146915 non-null  datetime64[ns]
 4   location              144910 non-null  object        
 5   auction_title         146914 non-null  object        
 6   auction_url           0 non-null       object        
 7   lot_number            146914 non-null  object        
 8   currency              146915 non-null  object        
 9   ah_low_estimate       146915 non-null  float64       
 10  ah_high_estimate      146915 non-null  float64       
 11  ah_bp_price           146915 non-null  float64       
 12  descript

## Cleaning of SoT data

In [6]:
# 6. Cleaning of SoT data
cleaner = (
    SoTDataCleaner(final_df)
        .drop_duplicate_lots()
        .drop_na_prices_and_dates()
        .normalize_sale_date()
        .flag_price_outliers(threshold=2)
)


[1] Drop duplicate lots
  Rows: 146915 → 146915

[2] Drop rows with missing price or sale date
  Rows: 146915 → 146915

[3] Normalize sale_date and extract year
  Rows: 146915 → 146915 (no drops)

[4] Flag price outliers (threshold=2)
  Outliers flagged: 11356 / 146915


# II. Inspection

## Outlier Analysis of SoT data

In [7]:
# 7. Outlier Analysis of SoT data
top_outliers = cleaner.show_top_outliers(10)
display(top_outliers)

if remove_outliers_in_final:
    final_clean_df = (
        cleaner
            .remove_outliers()
            .get_df()
    )
    print("Outlier removal enabled: flagged outliers removed from final output.")
else:
    final_clean_df = cleaner.get_df().copy()
    print("Outlier removal disabled: flagged outliers kept in final output.")



[5] Show top 10 outliers


,ah_bp_price_usd,ah_low_estimate_usd,ah_high_estimate_usd,relative_diff
46241,57285000.00,0.0,0.0,inf
46240,32035000.00,0.0,0.0,inf
46207,28851582.01,0.0,0.0,inf
46150,21792431.83,0.0,0.0,inf
46153,93105000.00,0.0,0.0,inf
46264,26402500.00,0.0,0.0,inf
15586,11838367.92,0.0,0.0,inf
16264,32937500.00,0.0,0.0,inf
16029,1658998.00,0.0,0.0,inf
16291,74010000.00,0.0,0.0,inf


Outlier removal disabled: flagged outliers kept in final output.


## Display count by year and auction house

In [8]:
# 8. Display count by year and auction house
counts_wide = cleaner.counts_by_year_and_house()
display(counts_wide)


[6] Counts by year and auction house


auction_house,Christies,Phillips,Sotheby's
year,,,
1991,19,0,0
1992,104,0,0
1993,119,0,0
1994,131,0,0
1995,190,0,0
1996,665,0,0
1997,987,0,0
1998,850,0,0
1999,2102,0,0


In [9]:
monthly_counts_2023 = cleaner.count_by_month_for_year(2025)
display(monthly_counts_2023)


[7] Counts by month for year 2025
  Total rows in 2025: 1075


,count
month,
1,162
2,217
3,301
4,139
5,211
6,24
7,8
8,1
9,2


In [10]:
final_clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146915 entries, 0 to 146914
Data columns (total 43 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   asset_type            146915 non-null  object        
 1   auction_house         146915 non-null  object        
 2   sale_id               146914 non-null  object        
 3   sale_date             146915 non-null  datetime64[ns]
 4   location              144910 non-null  object        
 5   auction_title         146914 non-null  object        
 6   auction_url           0 non-null       object        
 7   lot_number            146914 non-null  object        
 8   currency              146915 non-null  object        
 9   ah_low_estimate       146915 non-null  float64       
 10  ah_high_estimate      146915 non-null  float64       
 11  ah_bp_price           146915 non-null  float64       
 12  description_block     146915 non-null  object        
 13 

# Save as JSON

In [11]:
# 9. Save as JSON

# columns to drop
cols_to_drop = ["relative_diff", "row_based_outlier", "midpoint", "year"]

# drop columns
final_clean_df = final_clean_df.drop(columns=cols_to_drop, errors="ignore")

# convert all values to string
final_clean_df = final_clean_df.astype(str)

file_name = f"Data/Output/{output_fname}.json"
# save to JSON
final_clean_df.to_json(
    file_name,
    orient="records",
    force_ascii=False,
    indent=4,
)

print(f"✅ Saved {len(final_clean_df)} rows to {file_name}")


✅ Saved 146915 rows to Data/Output/artwork_abs_dataset_sot.json
